# QICK's signal generator on the VRK160, under PYNQ

This drives QICK's `axis_signal_gen_v6` on an AMD Versal VRK160 through PYNQ,
and measures what comes back through the RF loop:

    axis_signal_gen_v6 -> RF-DAC -> XM855 balun -> RF-ADC -> capture -> DDR

Nothing here is a reimplementation. The generator is QICK's IP, driven through
QICK's own generated PYNQ driver, which PYNQ binds by VLNV
(`qick:ip:axis_signal_gen_v6:1.0`) when the overlay loads. What lives in
`sgloop.py` is only what QICK cannot know: which DMA feeds the envelope and how
the 160-bit waveform descriptor reaches `s1_axis` — in QICK that comes from the
tProcessor, and this design has none yet.

## Before running

The PDI is downloaded once per boot. **Versal RF devices do not support PL
Reload** (AMD's RF known issues), so a second download leaves the DAC tile off
with no way back but a reboot. `SgLoop()` detects a design that is already
loaded and attaches without downloading, so re-running these cells is safe.

In [ ]:
import sys, os

# sgloop.py, the PDI, the .hwh and the .dtbo live in /home/xilinx, while this
# notebook usually sits under /home/xilinx/jupyter_notebooks. Put the module
# on the path rather than keeping a second copy next to the notebook: two
# copies of a driver drift, and the one you are not looking at is the one
# that runs.
SGLOOP_DIR = os.environ.get("SGLOOP_DIR", "/home/xilinx")
if SGLOOP_DIR not in sys.path:
    sys.path.insert(0, SGLOOP_DIR)

import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from sgloop import SgLoop, freq2reg, pack_descriptor

sg = SgLoop()

## Reading a capture

Three properties of this bench have to be in the model or the numbers look
wrong.

**The ADC folds.** Its output is 3932.16 MSps complex, I on even words and Q on
odd, so everything folds into ±1966.08 MHz.

**Its real-to-I/Q conversion shifts by Fs/4.** The ADC's low-power mixer runs
in Real→I/Q mode at 1966.08 MHz, and that shift *is* the conversion — mix by
Fs/4, decimate by two, and the complex output sits there. It is not stray
configuration to be removed. `rfloop` hid it by giving the DAC an equal and
opposite −Fs/4; this overlay cannot, because its DAC runs Real→Real, where
PG443 requires the coarse mixer bypassed.

So the term is structural, and `sgloop.rf_to_capture()` carries it for you
rather than leaving it to be remembered at every measurement.

**And the XM855 balun passes 50 MHz to 6 GHz**, on the RF frequency, not the
measured one. A peak at 0 MHz is the ADC's own offset and never something that
came round the loop.

In [ ]:
from sgloop import FS_C, FS_DAC, ADC_SHIFT, rf_to_capture, capture_to_rf

print(f"  the ADC's real->I/Q conversion shifts by {ADC_SHIFT:.2f} MHz\n")
for f in (100, 300, 500, 700):
    print(f"  play({f:5.1f} MHz)  ->  appears at {rf_to_capture(f):8.2f} MHz")

print(f"\n  and backwards: a peak at 1466.08 could be {capture_to_rf(1466.08)}")

## One tone

`outsel=1` takes the DDS alone, leaving the envelope out of the path — the
simplest thing that exercises PS → GPIO → descriptor → generator → DAC → loop.

`mode=1` makes it periodic. It matters: a one-shot burst of `nsamp` beats lasts
a few microseconds and is long over by the time Python gets round to arming the
capture. Deterministic triggering is what the tProcessor is for.

In [ ]:
sg.play(freq=500.0, gain=20000, nsamp=2000, outsel=1, mode=1)

# capture_settled flushes the previous burst first: whatever the last reader
# left in cap_fifo comes back on the NEXT capture as if it were fresh, which
# shows up as a tone at the right frequency but far too weak.
w = sg.capture_settled()

peak_f, peak_db = sg.spectrum(w, n_peaks=1)[0]
print(f"  peak: {peak_f:.2f} MHz at {peak_db:.1f} dB over the floor")
print(f"  expected {rf_to_capture(500.0):.2f} MHz")
if peak_db < 40:
    print("  NOTE: that is weak. Expect 85-90 dB; re-run the cell.")

iq = w[0::2].astype(float) + 1j*w[1::2].astype(float)
sp = np.abs(np.fft.fftshift(np.fft.fft(iq*np.hanning(len(iq)))))
fr = np.fft.fftshift(np.fft.fftfreq(len(iq), d=1/FS_C))
db = 20*np.log10(sp/np.median(sp))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(fr, db, lw=0.7)
for s in (-1, 1):
    a1.axvline(s*rf_to_capture(500.0), color='crimson', ls='--', lw=1)
a1.set_xlabel('MHz'); a1.set_ylabel('dB over the noise floor')
a1.set_title('full span'); a1.grid(alpha=.3)

# Zoomed: a single bin in 32768 is invisible across the full span.
m = np.abs(np.abs(fr) - rf_to_capture(500.0)) < 20
a2.plot(fr[m & (fr > 0)], db[m & (fr > 0)], lw=1)
a2.axvline(rf_to_capture(500.0), color='crimson', ls='--', lw=1,
           label=f'predicted {rf_to_capture(500.0):.2f}')
a2.set_xlabel('MHz'); a2.set_title('+/- 20 MHz around the prediction')
a2.legend(); a2.grid(alpha=.3)
plt.tight_layout()
plt.show()

## A sweep

The measured frequency against the model, over the range the balun passes.
Points on the line mean the generator is tuning and the loop is carrying it.

In [ ]:
requests = np.arange(100, 1500, 100.0)
measured, levels = [], []

for f in requests:
    sg.play(freq=f, gain=20000, nsamp=2000, outsel=1, mode=1)
    pk = sg.spectrum(n_peaks=1)[0]
    measured.append(pk[0]); levels.append(pk[1])

measured, levels = np.array(measured), np.array(levels)
expected = np.array([rf_to_capture(f) for f in requests])
err = measured - expected

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(requests, expected, '-',  color='crimson', label='model')
a1.plot(requests, measured, 'o', ms=5, label='measured')
a1.set_ylabel('MHz in the capture'); a1.legend(); a1.grid(alpha=.3)
a1.set_title('Requested frequency vs what comes back')

a2.plot(requests, err, 'o-', ms=4)
a2.axhline(0, color='k', lw=.5)
a2.set_xlabel('requested (MHz)'); a2.set_ylabel('error (MHz)')
a2.grid(alpha=.3)
plt.tight_layout()
plt.show()

print(f"  worst error {np.abs(err).max():.2f} MHz   "
      f"(FFT bin is {FS_C/len(iq):.3f} MHz)")
print(f"  level {levels.min():.1f} to {levels.max():.1f} dB over the floor")

## An envelope

`outsel=0` multiplies the DDS by the envelope, which is the shape QICK actually
uses for a pulse. The envelope is streamed into the generator's memory by DMA,
gated by `START_ADDR` and `WE` — the same sequence as QICK's
`AbsArbSignalGen.load()`, with I in bits [15:0] and Q in [31:16].

In [ ]:
# The envelope memory is read ONE ROW PER BEAT, and a row is N_DDS = 16
# samples, so an envelope of ENV_SAMPS samples lasts ENV_SAMPS/16 beats and
# nsamp -- which counts beats -- has to match it.
ENV_SAMPS = 2048
n = np.arange(ENV_SAMPS)
sigma = 200.0
env = 28000*np.exp(-((n - ENV_SAMPS/2)**2)/(2*sigma**2))
sg.load_envelope(env.astype(complex))

nsamp = ENV_SAMPS // 16
sg.play(freq=300.0, gain=20000, nsamp=nsamp, outsel=0, mode=1)
w = sg.capture_settled()

if w is None:
    print("  capture failed even after a retry -- re-run the cell.")
else:
    iq = w[0::2].astype(float) + 1j*w[1::2].astype(float)
    mag = np.abs(iq)

    # |iq| is not smooth here. A clean analytic signal would give a flat
    # magnitude for a constant envelope; this one oscillates, so the capture
    # is not a clean analytic pair at this carrier. A short moving average
    # recovers the shape -- 33 samples is far below the pulse width and far
    # above the carrier period.
    k = 33
    smooth = np.convolve(mag, np.ones(k)/k, mode='same')

    pk = int(np.argmax(smooth))
    lo, hi = max(0, pk - ENV_SAMPS), min(len(mag), pk + ENV_SAMPS)
    print(f"  capture {len(iq)} samples, smoothed peak {smooth[pk]:.0f} at {pk}")

    fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(13, 3.4))
    a1.plot(env); a1.set_title('envelope loaded'); a1.set_xlabel('sample')
    a1.grid(alpha=.3)

    a2.plot(smooth, lw=.6)
    a2.set_title('|captured|, smoothed -- pulse train')
    a2.set_xlabel('sample'); a2.grid(alpha=.3)

    a3.plot(np.arange(lo, hi), mag[lo:hi], lw=.4, alpha=.35, label='|iq|')
    a3.plot(np.arange(lo, hi), smooth[lo:hi], lw=1.5, label=f'{k}-sample mean')
    a3.set_title('one pulse'); a3.set_xlabel('sample')
    a3.legend(); a3.grid(alpha=.3)
    plt.tight_layout()
    plt.show()

## What this shows, and what it does not

**Shows.** PYNQ binds QICK's generated driver to QICK's IP by VLNV on Versal;
the descriptor and envelope paths work from Python; and the RF loop carries the
result at 85–90 dB over the noise floor with the frequency predicted to better
than an FFT bin.

**Does not.** There is no tProcessor, so nothing here is deterministically
timed — `mode=1` sidesteps that by leaving the generator running. And this is
not `QickSoc`: `import qick` still fails on this image, first on a hard PYNQ
version check (`requires PYNQ < 3.1`, this is 4.0.0) and then on `import
xrfdc`, which does not exist on Versal. Getting `AveragerProgramV2` and QICK's
own notebooks running needs an RFDC class over `xvrfdc` and is a separate job.